# CS103 Elements of AI Final Project

Read the project description carefully and follow all instructions. This project is an individual task: you may discuss ideas with others, but you must not work on solutions together or share code. Any form of plagiarism or unauthorized collaboration will result in an F for the entire course.

## Guideline

Primary TA: "" ()

**How to submit**
* Follow only the 🛑 <mark> TODO</mark> instructions. **Do not** modify other code unless a TODO explicitly asks for it.
* You don't need to submit the original file. Your score will be automatically posted on the LeaderBoard
* Due date: 23:59:59 **_____** (__). Late submission is accepted until 23:59:59 **_____** (__) with a 20% penalty.

**Before submitting**
* Restart the runtime and run all cells in order.
* Complete every 🛑 <mark> TODO</mark>.

**Restrictions**
* Use only the packages and resources provided for this homework unless additional tools are explicitly allowed.
* The recipe database used in the ScienceWorld section should be treated as an external resource for retrieval, not as a solution file to inspect manually.

## Overview

In final project, you will build LangGraph-based agent to solve the CS103ScienceWorld tasks.

## Note

* You will need to use VectorDB. We will give you embedded vectors for the recipe corpus so you don't need to embed all texts.
* `BAAI/bge-m3` should be used as embedding model to encode query.
* The tasks of the final project is complete different from the original task. So you can't refer to the solution of the original tasks. Rather, we will give you `tiny` and `seen` tasks to practice them.

In [1]:
import os
import re
from typing import Any, Dict, List, Optional, Sequence, TypedDict

from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

from cs103_scienceworld import CS103ScienceWorldFinalProjectEnv

C:\Users\fligh\Temporary_Sandbox\CS103_ScienceWorld\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here is the llm setting. You may attach tools with given `ChatOpenAI` object.

In [2]:
CHATOPENAI_KWARGS = {
    "model": "Qwen/Qwen3-30B-A3B-Instruct-2507",
    # "model": "Qwen/Qwen3-30B-A3B-Thinking",
    "temperature": 0.7,
    "base_url": "http://192.249.19.233:7000/v1",
    "timeout": 10,
    "max_retries": 0,
    "api_key": "LOCAL_DUMMY"
}

llm = ChatOpenAI(
    **CHATOPENAI_KWARGS,
)

Setup your `Env` Object and look what tasks are available

In [3]:
env = CS103ScienceWorldFinalProjectEnv(envStepLimit=40)

task_names = env.get_task_names()

print(len(task_names))
print(task_names)

8
['cook-mini', 'cook-seen', 'corrosion-mini', 'corrosion-seen', 'recipe-pipeline-tiny', 'recipe-pipeline-seen', 'corrode-circuit-tiny', 'corrode-circuit-seen']


Since you will need RAG to solve some tasks, the `CS103ScienceWorldFinalProjectEnv` provides corpus and its embeddings. Note that you should use `BAAI/bge-m3` for query embedding. The embeddings are type of `np.ndarray`.

In [4]:
corpus = env.get_corpus()
embeddings = env.get_corpus_embedding()

print(len(corpus))
print(corpus[0])
print(embeddings.shape)

50
Recipe card for bread.
Gather a metal pot, flour, and water.
Mix the water and flour in the metal pot until dough forms.
Heat the dough on a stove until it becomes bread.
Focus on the bread when it is ready.
(50, 1024)


Here is the sandbox for the `CS103ScienceWorldFinalProjectEnv`(Same as assignment 7). The detailed explanation of the usage of this library is in Assignment 7. Play with the new tasks with tiny and seen tasks.

In [14]:
env = CS103ScienceWorldFinalProjectEnv()

task_num = 0
var_num = 0
task_name = task_names[task_num]
env.load(task_name, var_num, simplificationStr="")

initial_observation, initial_dictionary = env.reset()
print(f"Starting {task_name}")
print(env.get_task_description())

userInputStr = "look around"
while (userInputStr != "exit"):
    if (userInputStr == "help"):
        print("Possible actions: ")
        for actionStr in env.get_possible_actions():
            print("\t" + str(actionStr))

    elif (userInputStr == "objects"):
        print("Possible objects (one referent listed per object): ")
        for actionStr in env.get_possible_objects():
            print("\t" + str(actionStr))

    elif (userInputStr == "valid"):
        print("Valid action-object combinations:")
        print(env.get_valid_action_object_combinations_with_templates())

    elif (userInputStr == 'goals'):
        print(env.get_goal_progress())

    else:
        # Send user input, get response
        observation, reward, isCompleted, info = env.step(userInputStr)
        score = info['score']
        print("\n" + observation)
        print("Reward: " + str(reward))
        print("Score: " + str(score))
        print("isCompleted: " + str(isCompleted))
        # print("info: " + str(info))

    print("'help' lists valid action templates, 'objects' lists valid objects, use <tab> to list valid actions. ")
    print("'goals' lists progress on subgoals.")
    print("type 'exit' to quit.")

    userInputStr = input('> ')

Starting cook-mini
Your task is to heat the potato until it becomes baked potato. Everything you need is in this room. Use the stove, wait until the change happens, then focus on the baked potato.

This room is called the test kitchen. In it, you see: 
	the agent
	a substance called air
	a potato
	a stove, which is turned off. On the stove is: nothing.
You also see:
	nothing
Reward: 0
Score: 8
isCompleted: False
'help' lists valid action templates, 'objects' lists valid objects, use <tab> to list valid actions. 
'goals' lists progress on subgoals.
type 'exit' to quit.


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "C:\Users\fligh\Temporary_Sandbox\CS103_ScienceWorld\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3747, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\fligh\AppData\Local\Temp\ipykernel_32176\335799859.py", line 45, in <module>
    userInputStr = input('> ')
  File "C:\Users\fligh\Temporary_Sandbox\CS103_ScienceWorld\.venv\Lib\site-packages\ipykernel\kernelbase.py", line 1403, in raw_input
    return self._input_request(
           ~~~~~~~~~~~~~~~~~~~^
        str(prompt),
        ^^^^^^^^^^^^
    ...<2 lines>...
        password=False,
        ^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\fligh\Temporary_Sandbox\CS103_ScienceWorld\.venv\Lib\site-packages\ipykernel\kernelbase.py", line 1448, in _input_request
    raise KeyboardInterrupt(msg) from None
KeyboardInterrupt: Interrupted by user

Duri

KeyboardInterrupt: Interrupted by user

🛑 <mark> TODO</mark> <br>
You can freely define your `Node`, `Tool`, `Graph` object. Also you are allowed to use `Chroma` to solve the retrieval task. We will use your graph object directly for grading unseen tasks. For grading, you **MUST** use `StateGraph` type in `LangGraph`.

### Some Helpful Facts
- When the agent **focus on** wrong thing, the score changes into -100 regardless of current score and the `completed` will be set as `True`. If you terminate your loop by only seeing the `completed` flag, you may get extremely low score. It is okay to retry the task within your agentic loop.
- Your hand-crafted skill will help you to solve the task.

In [8]:
# Your Section #
from pathlib import Path

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from cs103_scienceworld import evaluate_final_project_tasks, grade_final_project_unseen_tasks

# Manual skill notes: these are short hints students can inspect and the agent can retrieve.
MANUAL_SKILL_NOTES = [
    "Skill: Read the task description first and retrieve notes before acting when recipe or corrosion knowledge may matter.",
    "Skill: In recipe tasks, gather ingredients first and use the container or appliance named in the retrieved note.",
    "Skill: In circuit or corrosion tasks, search for the target component and focus on it only after the important state change happens.",
    "Skill: If the environment says the request is ambiguous, answer with action 0 until the ambiguity is resolved.",
]

# BAAI/bge-m3 is used for every query and manual skill note embedding.
# The corpus document embeddings are already provided by the env, so we insert them directly.
rag_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = Chroma(
    collection_name="scienceworld-final-project-notebook",
    embedding_function=rag_embeddings,
)
vector_store._collection.upsert(
    ids=[f"corpus-{idx}" for idx in range(len(corpus))],
    documents=corpus,
    embeddings=embeddings.tolist(),
    metadatas=[{"source": "corpus"} for _ in corpus],
)
vector_store._collection.upsert(
    ids=[f"skill-{idx}" for idx in range(len(MANUAL_SKILL_NOTES))],
    documents=MANUAL_SKILL_NOTES,
    embeddings=rag_embeddings.embed_documents(MANUAL_SKILL_NOTES),
    metadatas=[{"source": "manual-skill"} for _ in MANUAL_SKILL_NOTES],
)


@tool
def retrieve_notes(query: str, k: int = 4) -> str:
    """Retrieve ScienceWorld notes and manual skill hints for the current situation."""
    docs = vector_store.similarity_search(query, k=k)
    return "\n\n".join(doc.page_content for doc in docs) if docs else "<no relevant notes found>"


tools = [retrieve_notes]
llm_with_tools = llm.bind_tools(tools)


class AgentState(TypedDict, total=False):
    llm: Any
    env: Any
    student_id: str
    task_name: str
    variation_idx: int
    task_description: str
    observation: str
    info: Dict[str, Any]
    valid_actions: List[str]
    corpus: List[str]
    trajectory: List[Dict[str, Any]]
    thought: str
    action: str
    step_index: int
    score: int
    reward: int
    turn_count: int
    total_reward: int
    final_score: int
    completed: bool
    auto_resolve_ambiguity: bool
    bootstrap_attempts: int
    max_bootstrap_attempts: int
    blocked_focus_actions: List[str]
    focus_cooldown: int


def extract_text_content(content: Any) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                parts.append(item.get("text", ""))
            else:
                parts.append(str(item))
        return "\n".join(part for part in parts if part)
    return str(content)


def parse_action(valid_actions: Sequence[str], raw_text: str) -> Optional[str]:
    if not raw_text:
        return None

    lookup = {action.lower(): action for action in valid_actions}
    text = raw_text.strip()
    match = re.search(r"Action\s*:\s*(.+)", text, flags=re.IGNORECASE)
    if match:
        candidate = match.group(1).strip().strip("`").strip()
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    stripped = text.strip("`").strip()
    if stripped.lower() in lookup:
        return lookup[stripped.lower()]

    for line in text.splitlines():
        candidate = line.strip().strip("`").strip()
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    for action in valid_actions:
        if action.lower() in text.lower():
            return action
    return None


def fallback_action(valid_actions: Sequence[str]) -> str:
    ordered_prefixes = [
        'pick up', 'move', 'pour', 'mix', 'activate', 'open', 'go ', 'teleport to', 'look around', 'wait'
    ]
    for prefix in ordered_prefixes:
        for action in valid_actions:
            if action.lower().startswith(prefix):
                return action
    for action in valid_actions:
        if not action.lower().startswith('focus on'):
            return action
    return valid_actions[0]


# Node 1: ask the tool-bound LLM for the next action.
def reason_node(state: AgentState) -> AgentState:
    recent_steps = state.get('trajectory', [])[-5:]
    recent_trajectory = "\n".join(
        f"- step={step['index']} action={step['action']} score={step['score']} reward={step['reward']}"
        for step in recent_steps
    ) or "- <none>"

    messages = [
        SystemMessage(
            content=(
                "You are a ScienceWorld ReAct agent. "
                "Use the retrieve_notes tool when task knowledge may help. "
                "Do not repeat recently failed focus actions until the environment clearly changes. "
                "Reply with exactly two lines: Thought: <brief reasoning> and Action: <exact valid action>."
            )
        ),
        HumanMessage(
            content=(
                f"Task name: {state['task_name']}\n"
                f"Task description: {state['task_description']}\n"
                f"Observation:\n{state['observation']}\n\n"
                f"Info: {state['info']}\n\n"
                f"Recent trajectory:\n{recent_trajectory}\n\n"
                f"Blocked focus actions: {state.get('blocked_focus_actions', [])}\n\n"
                + "Valid actions:\n"
                + "\n".join(f"- {action}" for action in state['valid_actions'])
            )
        ),
    ]

    response = state['llm'].invoke(messages)
    tool_rounds = 0
    while getattr(response, 'tool_calls', None) and tool_rounds < 2:
        messages.append(response)
        for tool_call in response.tool_calls:
            if tool_call['name'] != 'retrieve_notes':
                continue
            tool_result = retrieve_notes.invoke(tool_call.get('args', {}))
            messages.append(ToolMessage(content=tool_result, tool_call_id=tool_call['id']))
        response = state['llm'].invoke(messages)
        tool_rounds += 1

    raw_output = extract_text_content(response.content)
    action = parse_action(state['valid_actions'], raw_output)
    blocked_focus_actions = set(state.get('blocked_focus_actions', []))
    focus_cooldown = int(state.get('focus_cooldown', 0))

    if action and action.lower().startswith('focus on'):
        if action in blocked_focus_actions or focus_cooldown > 0:
            action = None

    if action is None:
        safe_actions = [
            candidate for candidate in state['valid_actions']
            if candidate not in blocked_focus_actions and not candidate.lower().startswith('focus on')
        ]
        action = fallback_action(safe_actions or state['valid_actions'])

    return {'thought': raw_output, 'action': action}


# Node 2: execute the action, stop at the step limit, and restart the episode after a hard -100 failure.
def act_node(state: AgentState) -> AgentState:
    observation, reward, completed, info = state['env'].step(state['action'])
    trajectory = list(state.get('trajectory', []))
    trajectory.append({
        'index': len(trajectory),
        'action': state['action'],
        'observation': observation,
        'reward': int(reward),
        'score': int(info.get('score', 0)),
        'completed': bool(completed),
        'moves': int(info.get('moves', len(trajectory) + 1)),
        'auto_resolved': False,
    })

    total_reward = int(state.get('total_reward', 0)) + int(reward)
    final_score = int(info.get('score', 0))
    blocked_focus_actions = list(state.get('blocked_focus_actions', []))
    focus_cooldown = max(int(state.get('focus_cooldown', 0)) - 1, 0)

    if state.get('auto_resolve_ambiguity', True):
        while observation.startswith('Ambiguous request:') and len(trajectory) < state['env'].envStepLimit:
            observation, reward, completed, info = state['env'].step('0')
            total_reward += int(reward)
            final_score = int(info.get('score', 0))
            trajectory.append({
                'index': len(trajectory),
                'action': '0',
                'observation': observation,
                'reward': int(reward),
                'score': int(info.get('score', 0)),
                'completed': bool(completed),
                'moves': int(info.get('moves', len(trajectory) + 1)),
                'auto_resolved': True,
            })
            if completed:
                break

    hard_failed = bool(completed and final_score <= -100)
    bootstrap_attempts = int(state.get('bootstrap_attempts', 0))
    max_bootstrap_attempts = int(state.get('max_bootstrap_attempts', 2))

    if hard_failed and state['action'].lower().startswith('focus on') and state['action'] not in blocked_focus_actions:
        blocked_focus_actions.append(state['action'])

    if hard_failed and bootstrap_attempts < max_bootstrap_attempts:
        print(f'Hard failure detected at -100. Bootstrapping again (attempt {bootstrap_attempts + 1}/{max_bootstrap_attempts}).')
        observation, info = state['env'].reset()
        return {
            'observation': observation,
            'info': info,
            'valid_actions': state['env'].get_valid_action_object_combinations(),
            'trajectory': [],
            'step_index': 0,
            'turn_count': 0,
            'total_reward': 0,
            'final_score': int(info.get('score', 0)),
            'completed': False,
            'bootstrap_attempts': bootstrap_attempts + 1,
            'blocked_focus_actions': blocked_focus_actions,
            'focus_cooldown': 2,
        }

    if len(trajectory) >= state['env'].envStepLimit:
        completed = True

    return {
        'observation': observation,
        'info': info,
        'valid_actions': state['env'].get_valid_action_object_combinations(),
        'trajectory': trajectory,
        'step_index': len(trajectory),
        'turn_count': len(trajectory),
        'total_reward': total_reward,
        'final_score': final_score,
        'completed': bool(completed),
        'bootstrap_attempts': bootstrap_attempts,
        'blocked_focus_actions': blocked_focus_actions,
        'focus_cooldown': focus_cooldown,
    }


def route_after_step(state: AgentState) -> str:
    if state.get('completed', False):
        return 'done'
    if state.get('step_index', 0) >= state['env'].envStepLimit:
        return 'done'
    return 'continue'


def build_initial_graph_state() -> Dict[str, Any]:
    return {
        'trajectory': [],
        'thought': '',
        'bootstrap_attempts': 0,
        'max_bootstrap_attempts': 2,
        'blocked_focus_actions': [],
        'focus_cooldown': 0,
    }


graph = StateGraph(AgentState)
graph.add_node('reason', reason_node)
graph.add_node('act', act_node)
graph.add_edge(START, 'reason')
graph.add_edge('reason', 'act')
graph.add_conditional_edges('act', route_after_step, {'continue': 'reason', 'done': END})
state_graph = graph
state_graph


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 38267.95it/s]


## Minimal LangGraph Skeleton
This cell is a very small starter that only shows how to define `State`, `Node`, and a cyclic `StateGraph`. It does not use any DB, tool, or RAG component.


In [ ]:
class SkeletonState(TypedDict, total=False):
    env: Any
    observation: str
    info: Dict[str, Any]
    valid_actions: List[str]
    trajectory: List[Dict[str, Any]]
    action: str
    step_index: int
    total_reward: int
    final_score: int
    completed: bool


def skeleton_reason(state: SkeletonState) -> SkeletonState:
    action = state['valid_actions'][0] if state.get('valid_actions') else 'look around'
    return {'action': action}


def skeleton_act(state: SkeletonState) -> SkeletonState:
    observation, reward, completed, info = state['env'].step(state['action'])
    trajectory = list(state.get('trajectory', []))
    trajectory.append({
        'index': len(trajectory),
        'action': state['action'],
        'observation': observation,
        'reward': int(reward),
        'score': int(info.get('score', 0)),
        'completed': bool(completed),
        'moves': int(info.get('moves', len(trajectory) + 1)),
        'auto_resolved': False,
    })
    if len(trajectory) >= state['env'].envStepLimit:
        completed = True
    return {
        'observation': observation,
        'info': info,
        'valid_actions': state['env'].get_valid_action_object_combinations(),
        'trajectory': trajectory,
        'step_index': len(trajectory),
        'total_reward': int(state.get('total_reward', 0)) + int(reward),
        'final_score': int(info.get('score', 0)),
        'completed': bool(completed),
    }


def skeleton_route(state: SkeletonState) -> str:
    if state.get('completed', False):
        return 'done'
    if state.get('step_index', 0) >= state['env'].envStepLimit:
        return 'done'
    return 'continue'


skeleton_graph = StateGraph(SkeletonState)
skeleton_graph.add_node('reason', skeleton_reason)
skeleton_graph.add_node('act', skeleton_act)
skeleton_graph.add_edge(START, 'reason')
skeleton_graph.add_edge('reason', 'act')
skeleton_graph.add_conditional_edges('act', skeleton_route, {'continue': 'reason', 'done': END})
skeleton_graph


## Seen Smoke Test
Run this cell first. The baseline agent now handles `-100` failures inside its own graph loop by bootstrapping the episode again. Seen trajectories are saved under `output_dir`.


In [9]:
seen_report = evaluate_final_project_tasks(
    llm=llm_with_tools,
    state_graph=state_graph,
    env=env,
    student_id='20260000',
    task_names=['recipe-pipeline-tiny', 'corrode-circuit-seen'],
    variation_sample_count=1,
    initial_graph_state=build_initial_graph_state(),
    print_progress=True,
    output_dir=Path('artifacts/final_project_seen'),
)

print(seen_report)
print(f'seen score: {seen_report.total_score}/{seen_report.max_score}')


Hard failure detected at -100. Bootstrapping again (attempt 1/2).
Grading progress: 1/2 episodes completed
Grading progress: 2/2 episodes completed
Student 20260000 final project score: 38/200 (avg 19.00, completed 2/2, turns 80)
recipe-pipeline-tiny: 25/100 across variations [2]
corrode-circuit-seen: 13/100 across variations [7]
seen score: 38/200


## Important
The cell below runs actual unseen grading once. Inside `grade_final_project_unseen_tasks`, the env step limit is clamped to 50 even if the env was created with a larger value.


In [10]:
unseen_report = grade_final_project_unseen_tasks(
    llm=llm_with_tools,
    state_graph=state_graph,
    env=env,
    student_id='20260000',
    variation_sample_count=1,
    initial_graph_state=build_initial_graph_state(),
    print_progress=True,
)

print(unseen_report)
print(f'unseen score: {unseen_report.total_score}/{unseen_report.max_score}')


Grading progress: 1/4 episodes completed
Hard failure detected at -100. Bootstrapping again (attempt 1/2).
Hard failure detected at -100. Bootstrapping again (attempt 2/2).
Grading progress: 2/4 episodes completed
Hard failure detected at -100. Bootstrapping again (attempt 1/2).
Hard failure detected at -100. Bootstrapping again (attempt 2/2).
Grading progress: 3/4 episodes completed
Grading progress: 4/4 episodes completed
Student 20260000 final project score: -182/400 (avg -45.50, completed 4/4, turns 112)
Amber Badger: 7/100 across variations [10]
Calm Badger: -100/100 across variations [10]
Clever Badger: -100/100 across variations [7]
Brisk Badger: 11/100 across variations [7]
unseen score: -182/400
